In [ ]:
import csv
import requests
import time

# 🔑 API-ключ 2GIS
import os
API_KEY = os.getenv("DGIS_API_KEY", "")

# 🌐 URL API 2GIS
URL = "https://routing.api.2gis.com/carrouting/6.0.0/global"
HEADERS = {"Content-Type": "application/json"}
PARAMS = {"key": API_KEY}

# 📄 Чтение первых 10 строк из CSV
input_file = "points.csv"
output_file = "routes_time.csv"

points = []
with open(input_file, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        if i >= 10:
            break
        points.append({
            "address": row["address"],
            "lat": float(row["lat"]),
            "lon": float(row["lon"])
        })

# 🧾 Подготовка выходного CSV
with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(["from_address", "to_address", "duration_min"])

    # 🔁 Перебор всех пар точек
    for i, start in enumerate(points):
        for j, end in enumerate(points):
            if i == j:
                continue

            payload = {
                "points": [
                    {"lat": start["lat"], "lon": start["lon"], "type": "stop", "start": True},
                    {"lat": end["lat"], "lon": end["lon"], "type": "stop"}
                ],
                "route_mode": "fastest",
                "traffic_mode": "jam",
                "transport": "car",
                "locale": "ru_RU"
            }

            try:
                response = requests.post(URL, params=PARAMS, headers=HEADERS, json=payload)
                data = response.json()

                if "result" not in data or not data["result"]:
                    print(f"⚠️ Нет маршрута: {start['address']} -> {end['address']}")
                    continue

                # 🕒 Извлекаем минимальное время
                best_route = min(data["result"], key=lambda r: r.get("total_duration", float("inf")))
                duration_min = best_route["total_duration"] / 60

                print(f"✅ {start['address']} → {end['address']} : {duration_min:.1f} мин")

                writer.writerow([start["address"], end["address"], round(duration_min, 2)])

            except Exception as e:
                print(f"❌ Ошибка при обработке {start['address']} -> {end['address']}: {e}")

            # ⏳ Пауза между запросами (API rate-limit)
            time.sleep(0.3)

print(f"\n📁 Готово! Результаты сохранены в {output_file}")

In [9]:
import csv
import requests
import time

# 🔑 API-ключ 2GIS
import os
API_KEY = os.getenv("DGIS_API_KEY", "")

# 🌐 URL API 2GIS
URL = "https://routing.api.2gis.com/carrouting/6.0.0/global"
HEADERS = {"Content-Type": "application/json"}
PARAMS = {"key": API_KEY}

payload = {
    "points": [
        {"lat": 47.22818, "lon": 39.722313, "type": "stop", "start": True},
        {"lat":  47.225738, "lon": 39.717448, "type": "stop"}
    ],
    "route_mode": "fastest",
    "traffic_mode": "jam",
    "transport": "car",
    "locale": "ru_RU"
}

# payload = {
#     "start": {"lat":47.22818, "lon": 39.722313},  # Координаты начала маршрута (например, Москва)
#     "finish": {"lat": 47.225738, "lon": 39.717448},  # Координаты конца маршрута (например, Санкт-Петербург)
#     "routingType": "auto"  # Тип маршрута (автомобильный)
# }

response = requests.post(URL, params=PARAMS, headers=HEADERS, json=payload)
data = response.json()


In [10]:
print(data)

{'message': None, 'query': {'locale': 'ru_RU', 'points': [{'lat': 47.22818, 'lon': 39.722313, 'start': True, 'type': 'stop'}, {'lat': 47.225738, 'lon': 39.717448, 'type': 'stop'}], 'route_mode': 'fastest', 'traffic_mode': 'jam', 'transport': 'car'}, 'result': [{'algorithm': 'with traffic jams', 'begin_pedestrian_path': {'geometry': {'selection': 'LINESTRING(39.722313 47.228179, 39.722469 47.228114)'}}, 'end_pedestrian_path': {'geometry': {'selection': 'LINESTRING(39.717397 47.225727, 39.717448 47.225737)'}}, 'filter_road_types': ['highway'], 'id': '3585277973190056486', 'maneuvers': [{'comment': 'start', 'icon': 'start', 'id': '12456869858225794555', 'outcoming_path': {'distance': 42, 'duration': 12, 'geometry': [{'color': 'ignore', 'length': 42, 'selection': 'LINESTRING(39.722469 47.228114, 39.722476 47.228120, 39.722488 47.228126, 39.722785 47.228190, 39.722796 47.228190, 39.722805 47.228190, 39.722811 47.228186, 39.722819 47.228178, 39.722828 47.228157, 39.722872 47.228062)', 'style